[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/07_batchnorm.ipynb)

# 🟡 Medium: Implement BatchNorm

Implement **Batch Normalization** with both **training** and **inference** behavior.

In training mode, use **batch statistics** and update running estimates:

$$\text{BN}(x) = \gamma \cdot \frac{x - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}} + \beta$$

where $\mu_B$ and $\sigma_B^2$ are the mean and variance computed **across the batch** (dim=0).

In inference mode, use the provided **running mean/var** instead of current batch stats.

### Signature
```python
def my_batch_norm(
    x: torch.Tensor,
    gamma: torch.Tensor,
    beta: torch.Tensor,
    running_mean: torch.Tensor,
    running_var: torch.Tensor,
    eps: float = 1e-5,
    momentum: float = 0.1,
    training: bool = True,
) -> torch.Tensor:
    # x: (N, D) — normalize each feature across all samples in the batch
    # running_mean, running_var: updated in-place during training; used as-is during inference
```

### Rules
- Do **NOT** use `F.batch_norm`, `nn.BatchNorm1d`, etc.
- Compute batch mean and variance over `dim=0` with `unbiased=False`
- Update running stats like PyTorch: `running = (1 - momentum) * running + momentum * batch_stat`
- Use `running_mean` / `running_var` for inference when `training=False`
- Must support autograd w.r.t. `x`, `gamma`, `beta`（running statistics 应视作 buffer，而不是需要梯度的参数）

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch

In [3]:
# ✏️ YOUR IMPLEMENTATION HERE

# def my_batch_norm(
#     x,
#     gamma,
#     beta,
#     running_mean,
#     running_var,
#     eps=1e-5,
#     momentum=0.1,
#     training=True,
# ):
#     """
#     A simple implementation of batch normalization.

#     Args:
#         x: Input tensor of shape (N, D)
#         gamma: Scale parameter of shape (D,)
#         beta: Shift parameter of shape (D,)
#         running_mean: Running mean of shape (D,)
#         running_var: Running variance of shape (D,)
#         eps: Small constant for numerical stability
#         momentum: Momentum for updating running statistics
#         training: Whether the layer is in training mode

#     Returns:
#         Normalized tensor of shape (N, D)
#     """
#     if training:
#         # Compute mean and variance from the current batch
#         batch_mean = torch.mean(x, dim=0)  # (D,)
#         batch_var = torch.var(x, dim=0, unbiased=False)  # (D,)

#         # Update running statistics
#         running_mean.mul_(1 - momentum).add_(momentum * batch_mean)
#         running_var.mul_(1 - momentum).add_(momentum * batch_var)

#         # Normalize the batch
#         x_normalized = (x - batch_mean) / torch.sqrt(batch_var + eps)
#     else:
#         # Normalize using running statistics
#         x_normalized = (x - running_mean) / torch.sqrt(running_var + eps)

#     # Scale and shift
#     out = gamma * x_normalized + beta
#     return out

In [4]:
def my_batch_norm(
    x,
    gamma,
    beta,
    running_mean,
    running_var,
    eps=1e-5,
    momentum=0.1,
    training=True,
):
    if training:
        batch_mean = x.mean(dim=0)  # (D,)
        batch_var = x.var(dim=0, unbiased=False)  # (D,) unbiased=False for population variance

        # update running statistics
        # running_stats = (1 - momentum) * running_stats + momentum * batch_stats
        with torch.no_grad():
            running_mean.mul_(1 - momentum).add_(momentum * batch_mean)
            running_var.mul_(1 - momentum).add_(momentum * batch_var)

        x_normalized = (x - batch_mean) / torch.sqrt(batch_var + eps)
    else:
        x_normalized = (x - running_mean) / torch.sqrt(running_var + eps)

    output = gamma * x_normalized + beta
    return output

In [5]:
# 🧪 Debug
x = torch.randn(8, 4)
gamma = torch.ones(4)
beta = torch.zeros(4)

# Running stats typically live on the same device and shape as features
running_mean = torch.zeros(4)
running_var = torch.ones(4)

# Training mode: uses batch stats and updates running_mean / running_var
out_train = my_batch_norm(x, gamma, beta, running_mean, running_var, training=True)
print("[Train] Output shape:", out_train.shape)
print("[Train] Column means:", out_train.mean(dim=0))   # should be ~0
print("[Train] Column stds: ", out_train.std(dim=0))    # should be ~1
print("Updated running_mean:", running_mean)
print("Updated running_var:", running_var)

# Inference mode: uses running_mean / running_var only
out_eval = my_batch_norm(x, gamma, beta, running_mean, running_var, training=False)
print("[Eval] Output shape:", out_eval.shape)

[Train] Output shape: torch.Size([8, 4])
[Train] Column means: tensor([ 2.9802e-08, -1.4901e-08, -2.9802e-08, -2.9802e-08])
[Train] Column stds:  tensor([1.0690, 1.0690, 1.0690, 1.0690])
Updated running_mean: tensor([-0.0393, -0.0530,  0.0168, -0.0050])
Updated running_var: tensor([0.9906, 1.1036, 0.9776, 1.0411])
[Eval] Output shape: torch.Size([8, 4])


In [6]:
# ✅ SUBMIT
from torch_judge import check
check("batchnorm")


🧪 Testing: Implement BatchNorm (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] Training mode — zero mean per feature (4.1ms)
  ✅ [2/4] Training mode — numerical correctness and running stats update (3.2ms)
  ✅ [3/4] Inference mode — uses running statistics (1.8ms)
  ✅ [4/4] Gradient flow w.r.t inputs and affine params (29.1ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (38.3ms total)
  Progress saved. Run status() to see your dashboard.

